# 第67章 交互柱状图（px.bar）

<!-- module-learning-arc:start -->
> **Plotly 模块主线｜第 4 / 18 步：交互探索趋势、类别和变量关系**
>
> **持续应用背景：** 准备周度经营预警会：让读者通过悬停、缩放、下钻和层级探索，沿着异常、定位、行动的路径完成追问。
>
> **承接上一阶段：** 交互折线图（px.line）  →  **本章任务：** 交互柱状图（px.bar）  →  **下一步：** 交互散点图（px.scatter）
>
> **大作业连接：** 本章练习将成为《周度经营预警会：交互诊断与行动看板》的一部分，最终需要把诊断和行动视图组织成支持经营预警决策的可分享 HTML。
<!-- module-learning-arc:end -->


## 本章场景

比较不同类别谁大谁小、构成怎么分摊，是最常见也最需要表达的分析问题。


## 本章目标

学完本章，你将能够：

- **理解**：理解「交互柱状图（px.bar）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「交互柱状图（px.bar）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「交互柱状图（px.bar）」并读出其中的结论。


## 适用场景

**背景引入**：比较不同类别谁大谁小、构成怎么分摊，是最常见也最需要表达的分析问题。下面用交互柱状图（px.bar），把一串静态数字变成能悬停读值、点图例筛选、拖拽排序的图形，一眼就能看出哪个区域卖得最好、哪个渠道拖了后腿。它特别适合零基础同学先建立“字段 → 图形编码”的直觉——把一列数字映射到柱子的长度，再配齐标题和单位，图就能自己说话。（好比超市货架：x 是不同品类的标签，彼此不挨着、是离散的，柱长就是每类货的销量；几类货怎么摆由 barmode 决定——group 并排各比各，stack 从下往上叠看总和，relative 让正负都在中线两侧。）

比较类别数值并需要Hover、图例筛选或排序。


## 数据结构

类别列、数值列和可选分组列。


## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 barmode="group" 改为 barmode="stack" 或 "relative"，对比分组、堆积与相对堆积布局
2. 添加 text_auto=True 参数，观察柱形数值标签的显示效果
3. 修改 hovertemplate 自定义悬停信息格式，说明交互提示对精确读值的作用


## 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `regional.groupby()`、`px.bar()`、`fig.update_layout()`、`fig.show()` | 比较类别数值并需要Hover、图例筛选或排序。 | 类别顺序不稳定 |
| 进阶变体 | `px.bar()`、`fig.update_layout()`、`fig.show()` | 在基础图表上增加分组、注释、布局或交互 | 堆积图比较中间序列 |
| 关键参数 | `barmode` | group/stack/relative | 类别顺序不稳定 |
| 关键参数 | `orientation` | 方向 | 堆积图比较中间序列 |
| 关键参数 | `text_auto` | 标签 | 文字标签与Hover重复过多 |
| 关键参数 | `category_orders` | 顺序 | 类别顺序不稳定 |


## 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


<!-- math-foundation:chapter-67 -->
### 数学推导｜比较图中的差值与占比

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜差值回答“多了多少”。** $d_i=x_i-x_{ref}$，保留原单位。

**第 2 步｜比例回答“是基准的几倍”。** $r_i=x_i/x_{ref}$；相对变化是 $r_i-1$。

**第 3 步｜构成占比需要共同分母。** 令 $T=\sum_jx_j$，则 $s_i=x_i/T$，并且

$$
\sum_i s_i=\frac{\sum_i x_i}{T}=1
$$

所以只有互斥且穷尽的类别，才适合解释为整体构成。

**把上面的关系收束为本章计算式：**

$$
d_i=x_i-x_{ref},\qquad s_i=\frac{x_i}{\sum_jx_j}
$$

**符号解释：** $x_{ref}$ 是比较基准，$s_i$ 是类别 $i$ 的总体占比。

**代码对应：** 在绘图前计算差值或占比列，柱长只负责呈现已经定义好的指标。

**使用边界：** 排序、分母范围和是否包含“其他”类别都会改变占比解释。


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1️⃣ 数据导入：手建两个小表 + 读取三个公开数据集
funnel = pd.DataFrame(
    {
        "stage": ["访问", "查看商品", "加入购物车", "提交订单", "支付成功"],
        "users": [12000, 7200, 3100, 1850, 1420],
    }
)
timeline = pd.DataFrame(
    {
        "task": ["数据准备", "探索分析", "图表制作", "报告复核"],
        "start": pd.to_datetime(
            ["2026-03-01", "2026-03-04", "2026-03-08", "2026-03-12"]
        ),
        "finish": pd.to_datetime(
            ["2026-03-04", "2026-03-09", "2026-03-13", "2026-03-15"]
        ),
        "owner": ["数据", "分析", "分析", "负责人"],
    }
)
diamonds = pd.read_csv("/datasets/diamonds.csv")
flights = pd.read_csv("/datasets/flights.csv")
gapminder = pd.read_csv("/datasets/gapminder.csv")
print(
    f'Diamonds {len(diamonds):,} | Flights {len(flights):,} | Gapminder {len(gapminder):,} 行'
)


In [ ]:
# 2️⃣ 特征工程：为各章图表构造分析所需的派生字段
orders_full = diamonds.assign(
    date=pd.Timestamp("2026-01-01"),
    category=diamonds["cut"],
    region=diamonds["clarity"],
    channel=diamonds["color"],
    order_value=diamonds["price"],
    items=diamonds["carat"],
    sales=diamonds["price"],
    month="公开样本",
)
orders = orders_full.sample(5_000, random_state=55)

monthly = (
    flights.query("year == 1960")
    .rename(columns={"passengers": "sales"})
    .copy()
)
monthly["orders"] = monthly["sales"]
monthly["profit"] = monthly["sales"].rolling(3, min_periods=1).mean()

regional = orders_full.groupby(["region", "channel"], as_index=False)[
    "sales"
].sum()

hierarchy = (
    diamonds.groupby(["cut", "color"], as_index=False)["price"]
    .sum()
    .rename(
        columns={"cut": "department", "color": "category", "price": "sales"}
    )
)

countries = gapminder.query("year == 2007").assign(
    country=lambda frame: frame["country"],
    market=lambda frame: frame["country"],
    sales=lambda frame: frame["gdpPercap"],
    growth=lambda frame: frame["lifeExp"],
)
print(f"样本：orders {len(orders):,} | monthly {len(monthly):,} 行")


## 例 1｜最小可用图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
totals = (
    regional.groupby("region", as_index=False)["sales"]
    .sum()
    .sort_values("sales")
)
fig = px.bar(
    totals,
    x="sales",
    y="region",
    orientation="h",
    text_auto=True,
    title="区域总销售额",
)
fig.update_layout(xaxis_title="销售额（万元）", yaxis_title="区域")
fig.show()


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


**练一练**：上面的基础图表用的是横向条形（`orientation=\"h\"`）。请你把它改成纵向柱状图（`orientation=\"v\"`，并把 `x`、`y` 字段对调），再观察柱子的长短顺序有什么变化；想一想，如果想把销售额从高到低排列，该在拿到 `regional` 后先做哪一个排序操作？在下面单元格里填好你的版本，再运行答案单元格对照自检。


In [ ]:
try:
    pass
    # 请在下方填写代码
    # 需求：绘制各 region 的 sales 纵向柱状图，并尝试更换一个图表参数观察变化。
    # 提示：regional 含 region/channel/sales 三列，请先聚合各 region 的总销售额。

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 例 2｜进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
fig = px.bar(
    regional,
    x="region",
    y="sales",
    color="channel",
    barmode="group",
    text_auto=True,
    title="区域渠道销售对比",
)
fig.update_layout(
    xaxis_title="区域", yaxis_title="销售额（万元）", legend_title="渠道"
)
fig.show()


## 参数说明

- barmode：group/stack/relative
- orientation：方向
- text_auto：标签
- category_orders：顺序


## 结果解读

比较共享基线上的长度；通过图例暂时隐藏分组帮助检查。


## 本章实训：交互图与信息层次

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import plotly.express as px

report = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北", "西南"],
        "sales": [320, 250, 280, 190],
    }
)
_demo_fig = px.bar(report, x="region", y="sales", title="地区销售额")
_demo_fig.show()


### 第一个结果怎么读

Plotly 的基本流程是：准备表格、映射字段、设置标题、显示图形。悬停提示只能补充信息，不能替代坐标轴和单位。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
_demo_fig = px.bar(
    report.sort_values("sales", ascending=False),
    x="region",
    y="sales",
    text="sales",
    title="按销售额排序的地区销售额",
)
_demo_fig.update_traces(textposition="outside")
_demo_fig.update_layout(yaxis_title="销售额", xaxis_title="地区")
_demo_fig.show()


### 第二个结果怎么读

第二个实验增加数值标签并排序。请检查：标签是否遮挡、标题是否准确、图形是否仍然能在窄屏阅读。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：空数据还能不能画图

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import plotly.express as px

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
if report.empty:
    print("没有可绘制的数据，请先检查筛选条件。")
else:
    fig = px.bar(report, x="region", y="sales", title="地区销售额")
    fig.update_layout(yaxis_title="销售额", xaxis_title="地区")
    fig.show()


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

筛选后先判断是否为空，再调用绘图函数。空表不是绘图库的问题，而是上游筛选口径需要检查。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- 类别顺序不稳定
- 堆积图比较中间序列
- 文字标签与Hover重复过多


## 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 独立迁移练习

在默认图可读的前提下，增加一个 hover 字段或筛选交互。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    # 独立迁移练习：用 color 按区域着色，突出最高与最低
    # 【目标】给柱子加颜色维度，练习用颜色强调要对比的对象。
    import plotly.express as px

    # 起点示例(已可运行)：加 color="region" 给每根柱子着色。
    totals = (
        regional.groupby("region", as_index=False)["sales"]
        .sum()
        .sort_values("sales")
    )
    fig = px.bar(
        totals,
        x="sales",
        y="region",
        orientation="h",
        text_auto=True,
        color="region",
        color_discrete_sequence=px.colors.qualitative.Set2,
        title="区域总销售额（分色）",
    )
    fig.update_layout(
        xaxis_title="销售额（万元）", yaxis_title="区域", showlegend=False
    )
    fig.show()

    # ---- 反思记录：着色后，最高/最低区域是否更醒目 ----
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(f"改动：{change_note}")
    print(f"预期：{expected_change}")
    print(f"观察：{observed_change}")

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 小结

用交互柱状图比较类别、分组和堆积构成。


### 你已经掌握

- 判断交互柱状图（px.bar）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `barmode` | group/stack/relative |
| `orientation` | 方向 |
| `text_auto` | 标签 |
| `category_orders` | 顺序 |


### 需要注意

- 类别顺序不稳定
- 堆积图比较中间序列
- 文字标签与Hover重复过多


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


In [ ]:
# 参考答案：把 orientation 换成 "v"，并把 x / y 字段对调，柱体改为竖向排列。
region_agg = regional.groupby("region", as_index=False)["sales"].sum()
fig = px.bar(
    region_agg,
    orientation="v",
    x="region",
    y="sales",
    title="区域总销售额（纵向）",
)
fig.update_layout(xaxis_title="区域", yaxis_title="销售额（万元）")


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
fig = px.bar(
    regional,
    x="region",
    y="sales",
    color="channel",
    barmode="stack",
    title="区域渠道销售构成",
)
fig.update_traces(
    hovertemplate="%{x}<br>%{fullData.name}: %{y} 万元<extra></extra>"
)
fig.update_layout(
    xaxis_title="区域", yaxis_title="销售额（万元）", legend_title="渠道"
)
fig.show()
